# 18 — Retries, Backfills & Operational Semantics

### Why this exists
Retries are a correctness problem, not merely a resilience feature. This lab compares unsafe append behavior with idempotent upsert behavior and models an overlapping backfill.

### Learning objectives
model attempts and writes; quantify duplicates; implement idempotency; reason about overlapping windows

### Prerequisite
Python dictionaries and sets

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/18-retries-backfills.svg)

> **Question to carry through the lab:** If the same task runs twice after a timeout, what prevents the output from being duplicated?

## Capability contract

**Capability:** C04 — Lineage Orchestration & Recovery  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Trace a failure through a workflow and design safe rerun or recovery behavior.


## Mental model
A retry repeats work after uncertain completion. Safety comes from making the side effect idempotent or deduplicated, not from assuming the first attempt failed.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S04 — Incremental and idempotent pipelines (DATA-1070)** → [`../work_simulations/04_acme_commerce_duplicate_rerun/README.md`](../work_simulations/04_acme_commerce_duplicate_rerun/README.md)
- **S06 — Retries and backfills (DATA-1094)** → [`../work_simulations/06_acme_commerce_retry_side_effect/README.md`](../work_simulations/06_acme_commerce_retry_side_effect/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
events=[{"id":"o1","amount":100},{"id":"o2","amount":50},{"id":"o1","amount":100}]
unsafe=[]
for e in events: unsafe.append(e)
print("unsafe rows",len(unsafe),"unique ids",len({e['id'] for e in unsafe}))

In [ ]:
safe={}
for e in events: safe[e["id"]]=e
print("idempotent rows",len(safe),"total",sum(e["amount"] for e in safe.values()))

In [ ]:
# A backfill overlaps the regular run.
regular={"2026-09-29":["o1","o2"]}
backfill={"2026-09-28":["o0","o1"],"2026-09-29":["o1","o2"]}
print("overlap",set(regular["2026-09-29"]) & set(backfill["2026-09-29"]))

In [ ]:
def dedupe_batch(batches):
    seen=set(); output=[]
    for batch in batches:
        for key in batch:
            if key not in seen: output.append(key); seen.add(key)
    return output
print(dedupe_batch([regular["2026-09-29"],backfill["2026-09-29"]]))

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## Your task

**Goal:** Choose a safe retry strategy for a non-idempotent side effect.

**Do this:** Compare an idempotency key, transactional outbox and deduplication. State the evidence you would require before choosing one.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## 8. Production bridge
Production requires retry budgets, idempotency keys, run metadata, replay safety and clear semantics for partial completion.

## 9. Independent challenge
Design a retry policy for a side effect that is not naturally idempotent. Compare idempotency key, transactional outbox, and deduplication; state what evidence you need before choosing.

### Evidence to keep
duplicate count, idempotent result, overlap evidence, and a retry/backfill decision

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.

---

**Check your work:** [Open the reference solution](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/18.md) — only after you have attempted the exercise.